# Final Project: One Anomaly, Defended
**IELE756 – Preparación y Análisis de Datos**

**Integrantes:** Felipe Alonso, Juan Costa

**Repositorio GitHub:** [https://github.com/juancostalarrain-netizen/-iele756-Quilicura-La-Reina-Tiltil](https://github.com/juancostalarrain-netizen/-iele756-Quilicura-La-Reina-Tiltil)

**Video:** `[AGREGAR LINK AL VIDEO]`

---
## La Anomalía

Tiltil (código 13125) presenta una tasa de egresos hospitalarios GRD de **69.79 por 10.000 habitantes** — prácticamente idéntica a su tasa ENO de 77.42 por 10.000. En todas las demás comunas de la Región Metropolitana el cociente GRD/ENO oscila entre 7 y 20; en Tiltil es **0.9**. Dado que los egresos hospitalarios son normalmente mucho más frecuentes que las notificaciones de enfermedades de declaración obligatoria, una razón menor que 1 es estructuralmente imposible bajo funcionamiento normal del sistema. La hipótesis central es que el GRD de Tiltil está casi completamente ausente porque esa base registra el **establecimiento de atención**, no el domicilio del paciente, y Tiltil no alberga ningún hospital de mediana o alta complejidad: sus residentes se hospitalizan en establecimientos de Santiago u otras comunas, donde el egreso queda registrado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# Sube por la jerarquía hasta encontrar la carpeta output/ (funciona desde root o desde notebooks/)
BASE = Path.cwd()
while not (BASE / 'output').exists() and BASE != BASE.parent:
    BASE = BASE.parent
OUT      = BASE / 'output'
FIGS_DIR = BASE / 'figs'
FIGS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(OUT / 'tarea3_analytical_table.csv')
print(f'N comunas: {len(df)}')
print(f'BASE: {BASE}')
df[['nombre_comuna','pop_total','eno_rate_per_10k','grd_rate_per_10k']].head()

---
## Aislamiento de la anomalía

Calculamos el **cociente GRD/ENO** para cada comuna. Un valor < 1 indica que los egresos hospitalarios per cápita son menores que las notificaciones ENO per cápita — situación que no debería ocurrir bajo registro normal.

In [ ]:
# Cociente GRD/ENO por comuna (excluimos comunas con eno_rate == 0 para evitar divisiones)
df_valid = df[df['eno_rate_per_10k'] > 0].copy()
df_valid['grd_eno_ratio'] = df_valid['grd_rate_per_10k'] / df_valid['eno_rate_per_10k']

# Z-score de grd_rate_per_10k en la distribución regional
grd_mean = df_valid['grd_rate_per_10k'].mean()
grd_std  = df_valid['grd_rate_per_10k'].std()
df_valid['grd_zscore'] = (df_valid['grd_rate_per_10k'] - grd_mean) / grd_std

# Nuestras 3 comunas
our_codes = [13122, 13109, 13125]
cols_show = ['nombre_comuna', 'pop_total', 'eno_rate_per_10k', 'grd_rate_per_10k',
             'grd_eno_ratio', 'grd_zscore']

print('=== Nuestras 3 comunas ===')
print(df_valid[df_valid['codigo_comuna'].isin(our_codes)][cols_show].to_string(index=False))

print(f'\n=== Distribución regional ===')
print(f'GRD rate  —  media: {grd_mean:.1f}  |  mediana: {df_valid["grd_rate_per_10k"].median():.1f}  |  std: {grd_std:.1f}')
print(f'Ratio GRD/ENO  —  media: {df_valid["grd_eno_ratio"].mean():.2f}  |  mediana: {df_valid["grd_eno_ratio"].median():.2f}')

# ¿Cuántas comunas tienen ratio < 2?
n_low = (df_valid['grd_eno_ratio'] < 2).sum()
print(f'\nComunas con GRD/ENO < 2: {n_low} de {len(df_valid)}')
print(df_valid[df_valid['grd_eno_ratio'] < 2][cols_show].sort_values('grd_eno_ratio').to_string(index=False))

---
## Figura Principal (Headline Figure)

Panel izquierdo: tasa GRD vs tasa ENO para todas las comunas RM. El eje diagonal (ratio=1) es la línea de referencia imposible; todas las comunas deberían estar muy por encima. Panel derecho: cociente GRD/ENO por comuna, ordenado, con Tiltil destacado.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Panel izquierdo: scatter GRD vs ENO ──────────────────────────────────────
ax = axes[0]
our_mask    = df_valid['codigo_comuna'].isin(our_codes)
tiltil_mask = df_valid['codigo_comuna'] == 13125

ax.scatter(df_valid.loc[~our_mask, 'eno_rate_per_10k'],
           df_valid.loc[~our_mask, 'grd_rate_per_10k'],
           s=40, alpha=0.5, color='steelblue', label='Otras comunas RM')

for code, name, color in [(13122, 'Quilicura', 'darkorange'), (13109, 'La Reina', 'green')]:
    row = df_valid[df_valid['codigo_comuna'] == code]
    ax.scatter(row['eno_rate_per_10k'], row['grd_rate_per_10k'],
               s=120, color=color, zorder=5, label=name)
    ax.annotate(name, (row['eno_rate_per_10k'].values[0], row['grd_rate_per_10k'].values[0]),
                xytext=(8, 4), textcoords='offset points', fontsize=8, color=color)

row_t = df_valid[df_valid['codigo_comuna'] == 13125]
ax.scatter(row_t['eno_rate_per_10k'], row_t['grd_rate_per_10k'],
           s=200, color='firebrick', zorder=6, marker='*', label='Tiltil (anomalía)')
ax.annotate('Tiltil\n(GRD ≈ ENO)',
            (row_t['eno_rate_per_10k'].values[0], row_t['grd_rate_per_10k'].values[0]),
            xytext=(12, -30), textcoords='offset points', fontsize=9, color='firebrick',
            arrowprops=dict(arrowstyle='->', color='firebrick', lw=1.2))

xmax = df_valid['eno_rate_per_10k'].quantile(0.95)
xs = np.linspace(0, xmax, 100)
ax.plot(xs, xs, 'k--', linewidth=1, alpha=0.4, label='Ratio GRD/ENO = 1')
ax.set_xlabel('Tasa ENO por 10.000 hab.', fontsize=10)
ax.set_ylabel('Tasa GRD por 10.000 hab.', fontsize=10)
ax.set_title('Tasa GRD vs ENO\n(todas las comunas RM)', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(bottom=0)

# ── Panel derecho: cociente GRD/ENO ordenado ─────────────────────────────────
ax2 = axes[1]
df_sorted = df_valid.sort_values('grd_eno_ratio').reset_index(drop=True)
colors_bar = ['firebrick' if c == 13125 else
              ('darkorange' if c == 13122 else
               ('green' if c == 13109 else 'steelblue'))
              for c in df_sorted['codigo_comuna']]

ax2.barh(range(len(df_sorted)), df_sorted['grd_eno_ratio'], color=colors_bar, alpha=0.8)
ax2.axvline(1, color='black', linestyle='--', linewidth=1, alpha=0.6, label='Ratio = 1')
ax2.axvline(df_valid['grd_eno_ratio'].median(), color='grey', linestyle=':', linewidth=1,
            label=f'Mediana regional ({df_valid["grd_eno_ratio"].median():.1f})')

tiltil_pos = df_sorted[df_sorted['codigo_comuna'] == 13125].index[0]
ax2.set_yticks([tiltil_pos])
ax2.set_yticklabels(['Tiltil'], fontsize=9, color='firebrick')

patch_t = mpatches.Patch(color='firebrick', label='Tiltil')
patch_q = mpatches.Patch(color='darkorange', label='Quilicura')
patch_r = mpatches.Patch(color='green', label='La Reina')
patch_o = mpatches.Patch(color='steelblue', label='Otras comunas RM')
ax2.legend(handles=[patch_t, patch_q, patch_r, patch_o], fontsize=8)
ax2.set_xlabel('Cociente GRD / ENO (tasas por 10k)', fontsize=10)
ax2.set_title('Cociente GRD/ENO por comuna\n(ordenado ascendente)', fontsize=11)

plt.suptitle('Tiltil: egresos hospitalarios ausentes pese a carga de enfermedad normal',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'headline.png', bbox_inches='tight', dpi=150)
plt.savefig(OUT / 'headline.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura guardada en figs/headline.png')

---
## Chequeo de Explicación Alternativa 1: ¿Es un problema de subregistro en los datos brutos?

**Hipótesis**: El equipo 03 no capturó correctamente todos los registros GRD de Tiltil al procesar los datos en Tarea 2 (error de filtro, pérdida de filas). Si el total absoluto de egresos (`grd_total`) también es bajo en proporción a la población, el problema no es de procesamiento sino de codificación original.

**Lo que mostraría si fuera cierto**: `grd_total` de Tiltil sería incongruentemente pequeño. 
**Lo que mostró**: comparando el total absoluto de egresos de Tiltil (1.435 registros para ~205k habitantes en 5 años) con Quilicura (13.062 registros para ~236k habitantes), la proporción es 10:1 pese a poblaciones similares. El subregistro en el procesamiento no puede explicar una diferencia de ese orden de magnitud — si faltaran registros sería un error de décimas, no un factor 10.

In [ ]:
# Comparación de totales absolutos (no tasas) entre nuestras 3 comunas
our_df = df[df['codigo_comuna'].isin(our_codes)][[
    'nombre_comuna', 'pop_total', 'grd_total', 'eno_total',
    'grd_rate_per_10k', 'eno_rate_per_10k'
]].set_index('nombre_comuna')

print('=== Totales absolutos y tasas — nuestras 3 comunas ===')
print(our_df.to_string())

# Si el problema fuera de subregistro, esperaríamos:
#   grd_total_esperado ≈ (grd_rate_mediana_regional / 10000) * pop_total
grd_rate_median = df_valid['grd_rate_per_10k'].median()
tiltil_pop = df[df['codigo_comuna'] == 13125]['pop_total'].values[0]
tiltil_grd_obs = df[df['codigo_comuna'] == 13125]['grd_total'].values[0]
tiltil_grd_expected = (grd_rate_median / 10000) * tiltil_pop

print(f'\n=== Si Tiltil tuviera la tasa GRD mediana de la RM ({grd_rate_median:.0f}/10k) ===')
print(f'   GRD esperado:   {tiltil_grd_expected:,.0f} egresos')
print(f'   GRD observado:  {tiltil_grd_obs:,.0f} egresos')
print(f'   Factor faltante: {tiltil_grd_expected / tiltil_grd_obs:.1f}x')
print()
print('Conclusión: faltan ~{:.0f} egresos para alcanzar la tasa regional.'.format(
    tiltil_grd_expected - tiltil_grd_obs))
print('Un error de procesamiento no explica una diferencia de ese tamaño.')

---
## Chequeo de Explicación Alternativa 2: ¿Es el denominador poblacional el problema?

**Hipótesis**: La población de Tiltil (205.624) está sobreestimada en los datos del Censo 2024 para el código 13125 — quizás por una asignación incorrecta de microdatos a ese código de comuna. Si la población real de Tiltil es ~20.000-25.000 (su tamaño histórico como commune rural), la tasa GRD sería mucho mayor.

**Lo que mostraría si fuera cierto**: Con la población correcta (~22.000), la tasa GRD ascendería a valores más plausibles.
**Lo que mostró**: incluso con una población de 22.000 habitantes, la tasa GRD solo llegaría a ~650/10k — comparable a Quilicura. Sin embargo, la tasa ENO también colapsaría a ~720/10k (muy alta). El ratio GRD/ENO seguiría siendo ~0.9. La hipótesis del denominador infla ambas tasas por igual y **no resuelve la anomalía del cociente**. Además, la ENO rate no muestra el mismo problema: si la población estuviese mal, esperaríamos que ENO también fuera anómalamente baja, no normal.

In [ ]:
# ¿Qué pasaría si la población de Tiltil fuera ~22.000 (población histórica real)?
pop_alternativa = 22_000

tiltil_row = df[df['codigo_comuna'] == 13125].iloc[0]
grd_total_t = tiltil_row['grd_total']
eno_total_t = tiltil_row['eno_total']
pop_census  = tiltil_row['pop_total']

grd_rate_alt = (grd_total_t / pop_alternativa) * 10_000
eno_rate_alt = (eno_total_t / pop_alternativa) * 10_000
ratio_alt    = grd_rate_alt / eno_rate_alt

print('=== Sensibilidad al denominador poblacional ===')
print(f'Población Censo 2024 (código 13125): {pop_census:,.0f}')
print(f'Población alternativa (histórica):   {pop_alternativa:,.0f}')
print()
print(f'{'':20s}  {'Censo':>10s}  {'Alternativa':>12s}')
print(f'{'GRD rate /10k':20s}  {tiltil_row["grd_rate_per_10k"]:>10.1f}  {grd_rate_alt:>12.1f}')
print(f'{'ENO rate /10k':20s}  {tiltil_row["eno_rate_per_10k"]:>10.1f}  {eno_rate_alt:>12.1f}')
print(f'{'Ratio GRD/ENO':20s}  {tiltil_row["grd_rate_per_10k"]/tiltil_row["eno_rate_per_10k"]:>10.2f}  {ratio_alt:>12.2f}')
print()
print('Conclusión: el cociente GRD/ENO no cambia con el denominador — ambas tasas')
print('escalan igual. La anomalía no se explica por un error en el denominador poblacional.')

# Verificación adicional: ¿La tasa ENO de Tiltil es normal en la distribución regional?
eno_mean = df_valid['eno_rate_per_10k'].mean()
eno_std  = df_valid['eno_rate_per_10k'].std()
eno_z_tiltil = (tiltil_row['eno_rate_per_10k'] - eno_mean) / eno_std
grd_z_tiltil = (tiltil_row['grd_rate_per_10k'] - grd_mean) / grd_std

print(f'\nZ-score ENO de Tiltil: {eno_z_tiltil:+.2f}  (dentro del rango normal)')
print(f'Z-score GRD de Tiltil: {grd_z_tiltil:+.2f}  (extremo outlier negativo)')
print('→ El ENO de Tiltil es normal; sólo el GRD es anómalo. Coherente con hipótesis de codificación GRD.')

---
## Conclusión

La anomalía de Tiltil es estructural, no un artefacto de procesamiento ni de denominador. Su tasa ENO se ubica en el rango normal de la RM (z-score ≈ 0), lo que indica una carga de enfermedad comparable al resto de comunas. Su tasa GRD, en cambio, cae más de 8 desviaciones estándar por debajo de la media regional, con un cociente GRD/ENO de 0.9 cuando el resto de la RM registra valores entre 7 y 20.

La explicación más parsimoniosa es que la base GRD registra el **establecimiento de atención**, no el domicilio del paciente, y Tiltil carece de hospital propio. Sus residentes se hospitalizan en establecimientos de Santiago u otras comunas, donde el egreso queda contabilizado. Esto tiene implicancias de política pública directas: los indicadores de carga hospitalaria basados en GRD subestiman sistemáticamente la demanda generada por comunas rurales o peri-urbanas sin hospital, lo que puede llevar a una asignación de recursos orientada a donde están los hospitales, no a donde vive la población que los necesita.